# 🔬 Surgical Organ Classifier — PaliGemma 2 Fine-Tuning

Fine-tunes **PaliGemma 2** (`google/paligemma2-3b-pt-224`) for surgical organ
classification using laparoscopic images.  Supports multi-dataset training
with configurable per-dataset weights so you can prioritise domain-specific
data (e.g. DSAD for colorectal procedures).

**Recommended runtime:** GPU (T4 or A100)  
**Runtime → Change runtime type → GPU**

---

## Quick-start

1. Add `HF_TOKEN` to the Secrets panel (🔑) — needed to load PaliGemma 2.
2. Add `KAGGLE_USERNAME` + `KAGGLE_KEY` to Secrets — needed to download DSAD.
   Get your key at: **https://www.kaggle.com/settings/account** → API → Create New Token.
3. Run **Section 1** to install dependencies.
4. Run **Section 2** to clone your repo (or upload files manually).
5. Run **Section 3** to set up credentials and download DSAD.
6. Run **Section 4** to configure weights and train.
7. Run **Section 5** to evaluate and export.

---

## Dataset weight guide

| Dataset | Weight | Rationale |
|---------|--------|-----------|
| `dsad` | `3.0` | Colorectal-specific, expert pixel labels |
| `surgeon_corrections` | `5.0` | Your ground-truth annotations |
| `cholec80` | `1.0` | General laparoscopic, lower priority |
| `surgisr4k` | `0.0` | No organ labels — skip |

---

## DSAD source

DSAD is **not** on HuggingFace. It is hosted on:
- **Kaggle** (automated): `anindyamajumder/the-dresden-surgical-anatomy-dataset`
- **Figshare** (manual): https://springernature.figshare.com/articles/dataset/The_Dresden_Surgical_Anatomy_Dataset_for_abdominal_organ_segmentation_in_surgical_data_science/21702600

---
## 1 · Install dependencies

In [1]:
%%capture
!pip install -q \
    torch torchvision \
    transformers>=4.40.0 \
    peft \
    accelerate \
    datasets \
    Pillow \
    numpy \
    pycocotools \
    scikit-learn \
    matplotlib \
    tqdm \
    huggingface_hub \
    opencv-python-headless \
    kaggle

print('✓ Dependencies installed')

In [ ]:
# Quick patch
with open("/content/surgical-annotator/training/finetune.py", "r") as f:
    content = f.read()

if "import os" not in content[:500]:
    content = "import os\n" + content
    with open("/content/surgical-annotator/training/finetune.py", "w") as f:
        f.write(content)
    print("Patched - import os added")
else:
    print("import os already present - issue is elsewhere")

FileNotFoundError: [Errno 2] No such file or directory: '/content/surgical-annotator/training/finetune.py'

In [2]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠ No GPU detected. Switch runtime to GPU for reasonable training speed.')

Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB


---
## 2 · Load training code

Option A — clone from GitHub (recommended).  
Option B — upload `finetune.py`, `dataset.py`, `download_public_data.py` manually.

In [3]:
import os

# ── Configure ──────────────────────────────────────────────────────────
REPO_URL    = 'https://github.com/Vjp802/surgical-annotator'  # ← edit
REPO_BRANCH = 'main'
REPO_DIR    = '/content/surgical-annotator'
# ───────────────────────────────────────────────────────────────────────

if os.path.exists(REPO_DIR):
    print('Repo already cloned — pulling latest...')
    !git -C {REPO_DIR} pull
else:
    !git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}

# Add training/ to Python path so import works without package install
import sys
sys.path.insert(0, os.path.join(REPO_DIR, 'training'))
print('✓ Training code available')

Cloning into '/content/surgical-annotator'...
remote: Enumerating objects: 113, done.
remote: Counting objects: 100% (113/113), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 113 (delta 38), reused 105 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (113/113), 113.20 KiB | 14.15 MiB/s, done.
Resolving deltas: 100% (38/38), done.
✓ Training code available


In [14]:
with open('/content/surgical-annotator/training/finetune.py', 'r') as f:
    lines = f.readlines()

# Replace lines 465-468 directly
lines[464] = '    for name, param in model.named_parameters():\n'
lines[465] = '        if any(k in name for k in [\'vision_tower\', \'image_encoder\', \'vision_model\']):\n'
lines[466] = '            param.requires_grad = False\n'
lines[467] = '    for param in model.model.multi_modal_projector.parameters():\n'
lines[468] = '        param.requires_grad = False\n'

with open('/content/surgical-annotator/training/finetune.py', 'w') as f:
    f.writelines(lines)

print("✓ Fixed vision tower freeze")

# Verify
for i, line in enumerate(lines[462:471], start=463):
    print(f"{i}: {line}", end='')

✓ Fixed vision tower freeze
463: 
464:     # Freeze vision tower — only fine-tune the language model layers with LoRA
465:     for name, param in model.named_parameters():
466:         if any(k in name for k in ['vision_tower', 'image_encoder', 'vision_model']):
467:             param.requires_grad = False
468:     for param in model.model.multi_modal_projector.parameters():
469:         param.requires_grad = False
470:     # LoRA on language model attention layers
471:     lora_config = LoraConfig(


In [15]:
# ── Apply all finetune.py patches for PaliGemma 2 compatibility ──────
with open('/content/surgical-annotator/training/finetune.py', 'r') as f:
    content = f.read()

patches = 0

# 1. Add import os if missing
if 'import os\n' not in content[:200]:
    content = 'import os\n' + content
    patches += 1
    print("✓ Patch 1: import os added")

# 2. Fix vision tower freeze
old = '''    for param in model.vision_tower.parameters():
        param.requires_grad = False
    for param in model.multi_modal_projector.parameters():
        param.requires_grad = False'''
new = '''    for name, param in model.named_parameters():
        if any(k in name for k in ['vision_tower', 'image_encoder', 'vision_model']):
            param.requires_grad = False
    for param in model.model.multi_modal_projector.parameters():
        param.requires_grad = False'''
if old in content:
    content = content.replace(old, new)
    patches += 1
    print("✓ Patch 2: vision tower freeze fixed")

# 3. Add gradient checkpointing after get_peft_model
old = '    model = get_peft_model(model, lora_config)'
new = '''    model = get_peft_model(model, lora_config)
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()'''
if old in content and 'gradient_checkpointing_enable' not in content:
    content = content.replace(old, new)
    patches += 1
    print("✓ Patch 3: gradient checkpointing enabled")

# 4. Fix as_target_tokenizer
old = '''        with processor.tokenizer.as_target_tokenizer():
            targets = processor.tokenizer(
                label_names,
                return_tensors="pt",
                padding=True,
                add_special_tokens=False,
            )'''
new = '''        targets = processor.tokenizer(
            label_names,
            return_tensors="pt",
            padding=True,
            add_special_tokens=False,
        )'''
if old in content:
    content = content.replace(old, new)
    patches += 1
    print("✓ Patch 4: as_target_tokenizer removed")

# 5. Add <image> token to prompt
old = '''        inputs = processor(
            images=images,
            text=[PROMPT] * len(batch),
            return_tensors="pt",
            padding=True,
        )'''
new = '''        prompts = ["<image> " + PROMPT] * len(batch)
        inputs = processor(
            images=images,
            text=prompts,
            return_tensors="pt",
            padding=True,
        )'''
if old in content:
    content = content.replace(old, new)
    patches += 1
    print("✓ Patch 5: <image> token added to prompt")

# 6. Add token_type_ids to return dict
old = '''        return {
            "input_ids":        full_input_ids,
            "attention_mask":   full_attn_mask,
            "pixel_values":     inputs["pixel_values"],
            "labels":           labels,
            "label_names":      label_names,
        }'''
new = '''        token_type_ids = None
        if "token_type_ids" in inputs:
            answer_type_ids = torch.ones(
                len(batch), targets["input_ids"].shape[1], dtype=torch.long
            )
            token_type_ids = torch.cat(
                [inputs["token_type_ids"], answer_type_ids], dim=1
            )
        return {
            "input_ids":        full_input_ids,
            "attention_mask":   full_attn_mask,
            "pixel_values":     inputs["pixel_values"],
            "token_type_ids":   token_type_ids,
            "labels":           labels,
            "label_names":      label_names,
        }'''
if old in content:
    content = content.replace(old, new)
    patches += 1
    print("✓ Patch 6: token_type_ids added to collate")

# 7. Add token_type_ids to model forward call
old = '''            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                pixel_values=batch["pixel_values"],
                labels=batch["labels"],
            )'''
new = '''            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                pixel_values=batch["pixel_values"],
                token_type_ids=batch.get("token_type_ids"),
                labels=batch["labels"],
            )'''
if old in content:
    content = content.replace(old, new)
    patches += 1
    print("✓ Patch 7: token_type_ids added to model forward")

# 8. Add loss = outputs.loss
old = '''            )
            accelerator.backward(loss)'''
new = '''            )
            loss = outputs.loss
            accelerator.backward(loss)'''
if old in content and 'loss = outputs.loss' not in content:
    content = content.replace(old, new, 1)
    patches += 1
    print("✓ Patch 8: loss = outputs.loss added")

# 9. Fix evaluate -> evaluate_model
old = 'val_metrics = evaluate(model, processor, val_loader, accelerator, organ_list)'
new = 'val_metrics = evaluate_model(model, processor, val_loader, accelerator, organ_list)'
if old in content:
    content = content.replace(old, new)
    patches += 1
    print("✓ Patch 9: evaluate → evaluate_model fixed")

# 10. Fix stray triple quote if present
old = '        }\n    """\n    model.eval()'
new = '        }\n    return collate\n\n\ndef evaluate_model(model, processor, data_loader, device):\n    model.eval()'
if old in content:
    content = content.replace(old, new)
    patches += 1
    print("✓ Patch 10: stray triple quote removed")

with open('/content/surgical-annotator/training/finetune.py', 'w') as f:
    f.write(content)

print(f"\n✓ All done — {patches} patches applied")
print("Ready to train!")

✓ Patch 3: gradient checkpointing enabled
✓ Patch 4: as_target_tokenizer removed
✓ Patch 5: <image> token added to prompt
✓ Patch 6: token_type_ids added to collate
✓ Patch 7: token_type_ids added to model forward
✓ Patch 9: evaluate → evaluate_model fixed

✓ All done — 6 patches applied
Ready to train!


---
## 3 · Set up credentials & download DSAD

DSAD is hosted on **Kaggle**, not HuggingFace.  
The cell below reads `KAGGLE_USERNAME` and `KAGGLE_KEY` from Colab Secrets (🔑).  
To get your key: https://www.kaggle.com/settings/account → API → **Create New Token**.

### 3A · Configure credentials

In [5]:
# ── Configure ──────────────────────────────────────────────────────────
DSAD_MAX_SAMPLES = 2000   # Full dataset = 13 195; reduce for a quick test
DATASETS_ROOT    = '/content/datasets'
# ───────────────────────────────────────────────────────────────────────

import os
from google.colab import userdata

# ── Kaggle credentials (for DSAD download) ────────────────────────────
# DSAD lives on Kaggle: anindyamajumder/the-dresden-surgical-anatomy-dataset
# Steps to get your API key:
#   1. Visit https://www.kaggle.com/settings/account
#   2. API → Create New Token  (downloads kaggle.json)
#   3. Add KAGGLE_USERNAME and KAGGLE_KEY to the Colab Secrets panel (🔑)
try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
    print('✓ Kaggle credentials loaded from Secrets')
except Exception as e:
    print(f'⚠ Kaggle credentials not found in Secrets: {e}')
    print()
    print('Option A — add KAGGLE_USERNAME + KAGGLE_KEY to Colab Secrets (🔑)')
    print('Option B — upload kaggle.json manually:')
    print('  from google.colab import files; files.upload()  # select kaggle.json')
    print('  !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json')

# ── HuggingFace credentials (for PaliGemma 2 model weights) ──────────
try:
    import huggingface_hub
    huggingface_hub.login(token=userdata.get('HF_TOKEN'), add_to_git_credential=False)
    print('✓ HuggingFace credentials loaded')
except Exception as e:
    print(f'⚠ HuggingFace login skipped: {e}')
    print('  Add HF_TOKEN to Colab Secrets (🔑) if PaliGemma 2 download fails.')

✓ Kaggle credentials loaded from Secrets
✓ HuggingFace credentials loaded


### 3B · Download DSAD from Kaggle

In [4]:
!pip install kagglehub

In [6]:
import kagglehub
import os
import json
import shutil

# Download from Kaggle (uses cache if already downloaded this session)
print("Fetching DSAD from Kaggle...")
path = kagglehub.dataset_download(
    "anindyamajumder/the-dresden-surgical-anatomy-dataset"
)
print(f"✓ Kaggle cache: {path}")

# Source paths
src_base = f"{path}/Dresden Dataset"
src_images = f"{src_base}/images/train"
src_annots = f"{src_base}/annotations/instances_train.json"

# Pilot destination — stays in /content, no Drive copy needed
pilot_dir = "/content/datasets/dsad_pilot"
pilot_images_dir = f"{pilot_dir}/images"
os.makedirs(pilot_images_dir, exist_ok=True)

# Copy just 100 images
print("\nCopying 100 pilot images...")
all_images = sorted(os.listdir(src_images))[:100]
for f in all_images:
    shutil.copy(f"{src_images}/{f}", f"{pilot_images_dir}/{f}")
print(f"✓ Copied {len(all_images)} images")

# Load full COCO and filter to just those 100 images
print("Filtering COCO annotations...")
with open(src_annots) as f:
    coco = json.load(f)

pilot_filenames = set(all_images)
pilot_images_list = [
    img for img in coco["images"]
    if img["file_name"] in pilot_filenames
]
pilot_image_ids = {img["id"] for img in pilot_images_list}
pilot_annotations = [
    ann for ann in coco["annotations"]
    if ann["image_id"] in pilot_image_ids
]

pilot_coco = {
    "info": coco.get("info", {}),
    "licenses": coco.get("licenses", []),
    "images": pilot_images_list,
    "annotations": pilot_annotations,
    "categories": coco["categories"]
}

pilot_coco_path = f"{pilot_dir}/annotations.json"
with open(pilot_coco_path, "w") as f:
    json.dump(pilot_coco, f)

print(f"\n✓ Pilot dataset ready")
print(f"  Images:      {len(pilot_images_list)}")
print(f"  Annotations: {len(pilot_annotations)}")
print(f"  Categories:  {[c['name'] for c in pilot_coco['categories']]}")
print(f"  COCO JSON:   {pilot_coco_path}")

Fetching DSAD from Kaggle...


100%|██████████| 21.5G/21.5G [21:02<00:00, 18.3MB/s]

Extracting files...


✓ Kaggle cache: /root/.cache/kagglehub/datasets/anindyamajumder/the-dresden-surgical-anatomy-dataset/versions/5

Copying 100 pilot images...
✓ Copied 100 images
Filtering COCO annotations...

✓ Pilot dataset ready
  Images:      100
  Annotations: 257
  Categories:  ['abdominal_wall', 'colon', 'inferior_mesenteric_artery', 'intestinal_veins', 'liver', 'pancreas', 'small_intestine', 'spleen', 'stomach', 'ureter', 'vesicular_glands']
  COCO JSON:   /content/datasets/dsad_pilot/annotations.json


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Pilot setup script
# Dataset spec pointing at pilot
DATASET_SPECS = [
    '/content/datasets/dsad_pilot/annotations.json:'
    '/content/datasets/dsad_pilot/images:3.0',
]

# Reduced for pilot run
EPOCHS = 2
BATCH_SIZE = 4

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import kagglehub
import shutil

# Create folder structure on Drive
dsad_drive_dir = "/content/drive/MyDrive/surgical-annotator/datasets/dsad/raw"
os.makedirs(dsad_drive_dir, exist_ok=True)
print(f"✓ Created: {dsad_drive_dir}")

# Download from Kaggle
print("\nDownloading DSAD...")
path = kagglehub.dataset_download(
    "anindyamajumder/the-dresden-surgical-anatomy-dataset"
)
print(f"✓ Kaggle cache: {path}")

# Copy directly to Drive — no /content/ intermediary
print("\nCopying to Drive (this will take a few minutes)...")
shutil.copytree(path, dsad_drive_dir, dirs_exist_ok=True)
print("✓ Done!")

# Verify
items = os.listdir(dsad_drive_dir)
print(f"\nDrive contents ({len(items)} items):")
for item in sorted(items):
    print(f"  {item}")

Mounted at /content/drive
✓ Created: /content/drive/MyDrive/surgical-annotator/datasets/dsad/raw

✓ Kaggle cache: /root/.cache/kagglehub/datasets/anindyamajumder/the-dresden-surgical-anatomy-dataset/versions/5

Copying to Drive (this will take a few minutes)...


In [ ]:
import os
# Check how much has been copied so far
result = os.popen('du -sh /content/drive/MyDrive/surgical-annotator/datasets/dsad/raw/ 2>/dev/null').read()
print(f"Copied so far: {result.strip()}")

In [ ]:
import os

# Data already on Drive from previous session — no need to re-download
DATASETS_DIR = "/content/drive/MyDrive/surgical-annotator/datasets"
dsad_raw_dir = f"{DATASETS_DIR}/dsad/raw/Dresden Dataset"

# Verify it's there
if os.path.exists(dsad_raw_dir):
    print(f"✓ DSAD already on Drive at: {dsad_raw_dir}")
    print("\nContents:")
    for item in sorted(os.listdir(dsad_raw_dir)):
        print(f"  {item}")
else:
    print("✗ DSAD not found on Drive — re-run the kagglehub download cell")

DATASETS_ROOT = "/content/drive/MyDrive/surgical-annotator/datasets"

DATASET_SPECS = [
    f'{DATASETS_ROOT}/dsad/raw/Dresden Dataset/annotations/instances_train.json:'
    f'{DATASETS_ROOT}/dsad/raw/Dresden Dataset/images/train:3.0',
]

### 3C · Manual Figshare alternative

If Kaggle auth is unavailable, download DSAD manually from Figshare (free, no registration):

1. Go to: https://springernature.figshare.com/articles/dataset/The_Dresden_Surgical_Anatomy_Dataset_for_abdominal_organ_segmentation_in_surgical_data_science/21702600
2. Click **Download all** → save the ZIP.
3. Upload here and run the cell below.

In [ ]:
# ── Only run this if Kaggle download failed ───────────────────────────
# Upload the DSAD zip downloaded from Figshare, then run this cell.

from google.colab import files
import shutil, os

dsad_dir = f'{DATASETS_ROOT}/dsad'
os.makedirs(dsad_dir, exist_ok=True)

print('Select the DSAD zip file downloaded from Figshare:')
uploaded = files.upload()
for fname, data in uploaded.items():
    dest = os.path.join(dsad_dir, fname)
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'Uploaded {fname} → {dest}')
    print('The downloader will auto-unzip on next run.')

# Re-run the downloader — it will find the zip and process it
from download_public_data import download_dsad
dsad_coco       = download_dsad(DATASETS_ROOT, max_samples=DSAD_MAX_SAMPLES)
DSAD_IMAGES_DIR = f'{DATASETS_ROOT}/dsad/images'
print(f'\nDSAD COCO JSON : {dsad_coco}')

### 3D · Upload your own surgeon-corrected annotations (optional)

Export a COCO JSON from the annotation app, then upload it here.

In [ ]:
from google.colab import files
import shutil, os

OWN_ANNOTATIONS_DIR = '/content/own_annotations'
OWN_IMAGES_DIR      = '/content/own_images'
os.makedirs(OWN_ANNOTATIONS_DIR, exist_ok=True)
os.makedirs(OWN_IMAGES_DIR,      exist_ok=True)

print('Upload your COCO JSON export(s) and image archive (.zip) below.')
print('(Skip this cell if you have no surgeon-corrected data yet.)')

uploaded = files.upload()
for fname, data in uploaded.items():
    dest = os.path.join(OWN_ANNOTATIONS_DIR, fname)
    with open(dest, 'wb') as f:
        f.write(data)
    if fname.endswith('.zip'):
        shutil.unpack_archive(dest, OWN_IMAGES_DIR)
        print(f'  Extracted {fname} → {OWN_IMAGES_DIR}')
    else:
        print(f'  Saved {fname} → {dest}')

### 3E · (Optional) Download other public datasets

In [ ]:
# Uncomment datasets you want:

# from download_public_data import download_cholec80
# cholec_coco = download_cholec80(DATASETS_ROOT, max_samples=1000)

# from download_public_data import download_roboflow_surgical
# roboflow_coco = download_roboflow_surgical(DATASETS_ROOT, max_samples=500)

print('Uncomment the lines above to download additional datasets.')

---
## 4 · Configure training

In [7]:
import os
import torch

# Memory optimization
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Pilot dataset spec
DATASET_SPECS = [
    '/content/datasets/dsad_pilot/annotations.json:'
    '/content/datasets/dsad_pilot/images:3.0',
]

# Model
MODEL_ID   = 'google/paligemma2-3b-pt-224'
OUTPUT_DIR = '/content/checkpoints'

# Hyperparameters — reduced for T4 memory
EPOCHS       = 2
BATCH_SIZE   = 1
LR           = 2e-5
LORA_R       = 4
LORA_ALPHA   = 8
LORA_DROPOUT = 0.05
PATIENCE     = 3
TRAIN_RATIO  = 0.8
VAL_RATIO    = 0.1
BF16         = torch.cuda.is_available()

# Verify paths
for spec in DATASET_SPECS:
    parts = spec.split(':')
    ok_json = '✓' if os.path.exists(parts[0]) else '✗ MISSING'
    ok_imgs = '✓' if os.path.isdir(parts[1]) else '✗ MISSING'
    print(f'JSON {ok_json}  {parts[0]}')
    print(f'imgs {ok_imgs}  {parts[1]}')

print(f'\nBF16: {BF16} | Epochs: {EPOCHS} | Batch: {BATCH_SIZE} | LoRA r: {LORA_R}')

JSON ✓  /content/datasets/dsad_pilot/annotations.json
imgs ✓  /content/datasets/dsad_pilot/images

BF16: True | Epochs: 2 | Batch: 1 | LoRA r: 4


In [ ]:
import os
import torch

# ── Pilot dataset specs ───────────────────────────────────────────────
DATASET_SPECS = [
    '/content/datasets/dsad_pilot/annotations.json:'
    '/content/datasets/dsad_pilot/images:3.0',
]

# ── Model ─────────────────────────────────────────────────────────────
MODEL_ID   = 'google/paligemma2-3b-pt-224'
OUTPUT_DIR = '/content/checkpoints'

# ── Hyperparameters — reduced for pilot ──────────────────────────────
EPOCHS       = 2
BATCH_SIZE   = 4
LR           = 2e-5
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
PATIENCE     = 3
TRAIN_RATIO  = 0.8
VAL_RATIO    = 0.1
BF16         = torch.cuda.is_available()

# ── Verify paths exist ────────────────────────────────────────────────
for spec in DATASET_SPECS:
    parts      = spec.split(':')
    coco_json  = parts[0]
    images_dir = parts[1]
    weight     = float(parts[2]) if len(parts) > 2 else 1.0
    ok_json    = '✓' if os.path.exists(coco_json) else '✗ MISSING'
    ok_imgs    = '✓' if os.path.isdir(images_dir) else '✗ MISSING'
    print(f'  JSON {ok_json}  {coco_json}')
    print(f'  imgs {ok_imgs}  {images_dir}  (weight={weight})')
    print()

print(f'Output dir : {OUTPUT_DIR}')
print(f'BF16       : {BF16}')
print(f'Epochs     : {EPOCHS}')
print(f'Batch size : {BATCH_SIZE}')

  JSON ✓  /content/datasets/dsad_pilot/annotations.json
  imgs ✓  /content/datasets/dsad_pilot/images  (weight=3.0)

Output dir : /content/checkpoints
BF16       : True
Epochs     : 2
Batch size : 4


## 4B · Run training

In [8]:
# Build the finetune.py command
datasets_flags = ' '.join(f"'{spec}'" for spec in DATASET_SPECS)

cmd = (
    f"python {REPO_DIR}/training/finetune.py "
    f"  --datasets {' '.join(DATASET_SPECS)} "
    f"  --output_dir {OUTPUT_DIR} "
    f"  --model_id {MODEL_ID} "
    f"  --epochs {EPOCHS} "
    f"  --batch_size {BATCH_SIZE} "
    f"  --learning_rate {LR} "
    f"  --lora_r {LORA_R} "
    f"  --lora_alpha {LORA_ALPHA} "
    f"  --lora_dropout {LORA_DROPOUT} "
    f"  --patience {PATIENCE} "
    f"  --train_ratio {TRAIN_RATIO} "
    f"  --val_ratio {VAL_RATIO} "
    + ('  --bf16' if BF16 else '')
)

print('Command to run:')
print(cmd.replace('  ', '\n  '))

Command to run:
python /content/surgical-annotator/training/finetune.py
   --datasets /content/datasets/dsad_pilot/annotations.json:/content/datasets/dsad_pilot/images:3.0
   --output_dir /content/checkpoints
   --model_id google/paligemma2-3b-pt-224
   --epochs 2
   --batch_size 1
   --learning_rate 2e-05
   --lora_r 4
   --lora_alpha 8
   --lora_dropout 0.05
   --patience 3
   --train_ratio 0.8
   --val_ratio 0.1
   --bf16


In [ ]:
import os
for root, dirs, files in os.walk(
    "/content/drive/MyDrive/surgical-annotator/datasets/dsad/raw"
):
    level = root.replace(
        "/content/drive/MyDrive/surgical-annotator/datasets/dsad/raw", ""
    ).count(os.sep)
    if level > 3:
        continue
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if files:
        subindent = " " * 2 * (level + 1)
        for f in files[:3]:
            print(f"{subindent}{f}")
        if len(files) > 3:
            print(f"{subindent}... ({len(files)} total)")

raw/


In [ ]:
import shutil
import os

raw_dir = "/content/drive/MyDrive/surgical-annotator/datasets/dsad/raw/Dresden Dataset"
out_dir = "/content/datasets/dsad"

# Copy images
print("Copying images...")
shutil.copytree(
    f"{raw_dir}/images/train",
    f"{out_dir}/images",
    dirs_exist_ok=True
)
shutil.copytree(
    f"{raw_dir}/images/val",
    f"{out_dir}/images",
    dirs_exist_ok=True
)

# Copy the COCO annotation file — already exists!
print("Copying COCO annotations...")
shutil.copy(
    f"{raw_dir}/annotations/instances_train.json",
    f"{out_dir}/annotations.json"
)

print("Done!")
print(f"Images: {len(os.listdir(out_dir + '/images'))}")

Copying images...


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/surgical-annotator/datasets/dsad/raw/Dresden Dataset/images/train'

In [ ]:
with open('/content/surgical-annotator/training/finetune.py', 'r') as f:
    lines = f.readlines()

# Print lines around the error
for i, line in enumerate(lines[458:475], start=459):
    print(f"{i}: {line}", end='')

459:     model = PaliGemmaForConditionalGeneration.from_pretrained(
460:         args.model_id,
461:         torch_dtype=torch.bfloat16 if args.bf16 else torch.float32,
462:     )
463: 
464:     # Freeze vision tower — only fine-tune the language model layers with LoRA
465:     for param in model.vision_tower.parameters():
466:         param.requires_grad = False
467:     for param in model.multi_modal_projector.parameters():
468:         param.requires_grad = False
469: 
470:     # LoRA on language model attention layers
471:     lora_config = LoraConfig(
472:         r=args.lora_r,
473:         lora_alpha=args.lora_alpha,
474:         target_modules=["q_proj", "v_proj"],
475:         lora_dropout=args.lora_dropout,


In [ ]:
with open('/content/surgical-annotator/training/finetune.py', 'r') as f:
    content = f.read()

old = '''    # Freeze vision tower — only fine-tune the language model layers with LoRA
    for param in model.vision_tower.parameters():
        param.requires_grad = False
    for param in model.multi_modal_projector.parameters():
        param.requires_grad = False'''

new = '''    # Freeze vision tower — only fine-tune the language model layers with LoRA
    for name, param in model.named_parameters():
        if any(k in name for k in ['vision_tower', 'image_encoder', 'vision_model']):
            param.requires_grad = False
    for param in model.multi_modal_projector.parameters():
        param.requires_grad = False'''

content = content.replace(old, new)

with open('/content/surgical-annotator/training/finetune.py', 'w') as f:
    f.write(content)

print("✓ Patched vision tower freeze")

# Verify patch applied
with open('/content/surgical-annotator/training/finetune.py', 'r') as f:
    lines = f.readlines()
for i, line in enumerate(lines[458:475], start=459):
    print(f"{i}: {line}", end='')

✓ Patched vision tower freeze
459:     model = PaliGemmaForConditionalGeneration.from_pretrained(
460:         args.model_id,
461:         torch_dtype=torch.bfloat16 if args.bf16 else torch.float32,
462:     )
463: 
464:     # Freeze vision tower — only fine-tune the language model layers with LoRA
465:     for name, param in model.named_parameters():
466:         if any(k in name for k in ['vision_tower', 'image_encoder', 'vision_model']):
467:             param.requires_grad = False
468:     for param in model.multi_modal_projector.parameters():
469:         param.requires_grad = False
470: 
471:     # LoRA on language model attention layers
472:     lora_config = LoraConfig(
473:         r=args.lora_r,
474:         lora_alpha=args.lora_alpha,
475:         target_modules=["q_proj", "v_proj"],


In [ ]:
# Check actual model attribute names
from transformers import PaliGemmaForConditionalGeneration
import torch

model = PaliGemmaForConditionalGeneration.from_pretrained(
    'google/paligemma2-3b-pt-224',
    torch_dtype=torch.bfloat16
)

# Print top level attributes
print("Model attributes:")
for name, module in model.named_children():
    print(f"  {name}: {type(module).__name__}")

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

Model attributes:
  model: PaliGemmaModel
  lm_head: Linear


In [ ]:
for name, module in model.model.named_children():
    print(f"  {name}: {type(module).__name__}")

  vision_tower: SiglipVisionModel
  multi_modal_projector: PaliGemmaMultiModalProjector
  language_model: Gemma2Model


In [ ]:
with open('/content/surgical-annotator/training/finetune.py', 'r') as f:
    content = f.read()

old = '''    # Freeze everything except language model attention layers
    # Use name-based freezing to handle different transformers versions
    FREEZE_KEYWORDS = [
        'vision_tower', 'vision_model', 'image_encoder',
        'multi_modal_projector', 'mm_projector'
    ]
    frozen = 0
    for name, param in model.named_parameters():
        if any(k in name for k in FREEZE_KEYWORDS):
            param.requires_grad = False
            frozen += 1
    print(f"Froze {frozen} vision/projector parameter tensors")'''

new = '''    # Freeze vision tower and projector — only LoRA on language model
    for param in model.model.vision_tower.parameters():
        param.requires_grad = False
    for param in model.model.multi_modal_projector.parameters():
        param.requires_grad = False
    print("✓ Froze vision tower and multi modal projector")'''

content = content.replace(old, new)

with open('/content/surgical-annotator/training/finetune.py', 'w') as f:
    f.write(content)

print("✓ Patched — using model.model.vision_tower")

# Verify
with open('/content/surgical-annotator/training/finetune.py', 'r') as f:
    lines = f.readlines()
for i, line in enumerate(lines[458:478], start=459):
    print(f"{i}: {line}", end='')

✓ Patched — using model.model.vision_tower
459:     model = PaliGemmaForConditionalGeneration.from_pretrained(
460:         args.model_id,
461:         torch_dtype=torch.bfloat16 if args.bf16 else torch.float32,
462:     )
463: 
464:     # Freeze vision tower — only fine-tune the language model layers with LoRA
465:     for name, param in model.named_parameters():
466:         if any(k in name for k in ['vision_tower', 'image_encoder', 'vision_model']):
467:             param.requires_grad = False
468:     for param in model.multi_modal_projector.parameters():
469:         param.requires_grad = False
470: 
471:     # LoRA on language model attention layers
472:     lora_config = LoraConfig(
473:         r=args.lora_r,
474:         lora_alpha=args.lora_alpha,
475:         target_modules=["q_proj", "v_proj"],
476:         lora_dropout=args.lora_dropout,
477:         bias="none",
478:         task_type=TaskType.CAUSAL_LM,


In [ ]:
with open('/content/surgical-annotator/training/finetune.py', 'r') as f:
    lines = f.readlines()

# Print lines 460-475 to see exact current state
print("Current file state:")
for i, line in enumerate(lines[458:478], start=459):
    print(f"{i}: {line}", end='')

Current file state:
459:     model = PaliGemmaForConditionalGeneration.from_pretrained(
460:         args.model_id,
461:         torch_dtype=torch.bfloat16 if args.bf16 else torch.float32,
462:     )
463: 
464:     # Freeze vision tower — only fine-tune the language model layers with LoRA
465:     for name, param in model.named_parameters():
466:         if any(k in name for k in ['vision_tower', 'image_encoder', 'vision_model']):
467:             param.requires_grad = False
468:     for param in model.multi_modal_projector.parameters():
469:         param.requires_grad = False
470: 
471:     # LoRA on language model attention layers
472:     lora_config = LoraConfig(
473:         r=args.lora_r,
474:         lora_alpha=args.lora_alpha,
475:         target_modules=["q_proj", "v_proj"],
476:         lora_dropout=args.lora_dropout,
477:         bias="none",
478:         task_type=TaskType.CAUSAL_LM,


In [ ]:
with open('/content/surgical-annotator/training/finetune.py', 'r') as f:
    lines = f.readlines()

# Replace lines 468-469 (0-indexed: 467-468)
lines[467] = '    for param in model.model.multi_modal_projector.parameters():\n'
lines[468] = '        param.requires_grad = False\n'

with open('/content/surgical-annotator/training/finetune.py', 'w') as f:
    f.writelines(lines)

print("✓ Patched line 468")

# Verify
for i, line in enumerate(lines[462:472], start=463):
    print(f"{i}: {line}", end='')

✓ Patched line 468
463: 
464:     # Freeze vision tower — only fine-tune the language model layers with LoRA
465:     for name, param in model.named_parameters():
466:         if any(k in name for k in ['vision_tower', 'image_encoder', 'vision_model']):
467:             param.requires_grad = False
468:     for param in model.model.multi_modal_projector.parameters():
469:         param.requires_grad = False
470: 
471:     # LoRA on language model attention layers
472:     lora_config = LoraConfig(


In [ ]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 58.6 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
with open('/content/surgical-annotator/training/finetune.py', 'r') as f:
    lines = f.readlines()

for i, line in enumerate(lines[270:300], start=271):
    print(f"{i}: {line}", end='')

271:         images      = [item["image"] for item in batch]
272:         label_names = [item["label_name"] for item in batch]
273: 
274:         # --- Encode inputs (image + prompt) ---
275:         inputs = processor(
276:             images=images,
277:             text=[PROMPT] * len(batch),
278:             return_tensors="pt",
279:             padding=True,
280:         )
281: 
282:         # --- Encode targets (organ name strings) ---
283:         with processor.tokenizer.as_target_tokenizer():
284:             targets = processor.tokenizer(
285:                 label_names,
286:                 return_tensors="pt",
287:                 padding=True,
288:                 add_special_tokens=False,
289:             )
290: 
291:         # Build labels: -100 for all input positions, then answer tokens
292:         input_len  = inputs["input_ids"].shape[1]
293:         target_ids = targets["input_ids"]          # (B, T_ans)
294:         pad_id     = processor.tokenizer.pad_token_id
2

In [ ]:
with open('/content/surgical-annotator/training/finetune.py', 'r') as f:
    content = f.read()

# Fix 1: Remove as_target_tokenizer context manager (just tokenize directly)
old = '''        # --- Encode targets (organ name strings) ---
        with processor.tokenizer.as_target_tokenizer():
            targets = processor.tokenizer(
                label_names,
                return_tensors="pt",
                padding=True,
                add_special_tokens=False,
            )'''

new = '''        # --- Encode targets (organ name strings) ---
        targets = processor.tokenizer(
            label_names,
            return_tensors="pt",
            padding=True,
            add_special_tokens=False,
        )'''

content = content.replace(old, new)

# Fix 2: Add <image> token to prompt in processor call
old2 = '''        inputs = processor(
            images=images,
            text=[PROMPT] * len(batch),
            return_tensors="pt",
            padding=True,
        )'''

new2 = '''        # PaliGemma requires <image> token at start of text
        prompts = ["<image> " + PROMPT] * len(batch)
        inputs = processor(
            images=images,
            text=prompts,
            return_tensors="pt",
            padding=True,
        )'''

content = content.replace(old2, new2)

with open('/content/surgical-annotator/training/finetune.py', 'w') as f:
    f.write(content)

print("✓ Patched collate function")

# Verify both fixes
with open('/content/surgical-annotator/training/finetune.py', 'r') as f:
    lines = f.readlines()
for i, line in enumerate(lines[268:305], start=269):
    print(f"{i}: {line}", end='')

✓ Patched collate function
269:     """
270:     def collate(batch: list[dict]) -> dict:
271:         images      = [item["image"] for item in batch]
272:         label_names = [item["label_name"] for item in batch]
273: 
274:         # --- Encode inputs (image + prompt) ---
275:         # PaliGemma requires <image> token at start of text
276:         prompts = ["<image> " + PROMPT] * len(batch)
277:         inputs = processor(
278:             images=images,
279:             text=prompts,
280:             return_tensors="pt",
281:             padding=True,
282:         )
283: 
284:         # --- Encode targets (organ name strings) ---
285:         targets = processor.tokenizer(
286:             label_names,
287:             return_tensors="pt",
288:             padding=True,
289:             add_special_tokens=False,
290:         )
291: 
292:         # Build labels: -100 for all input positions, then answer tokens
293:         input_len  = inputs["input_ids"].shape[1]
294:         tar

In [ ]:
with open('/content/surgical-annotator/training/finetune.py', 'r') as f:
    lines = f.readlines()

for i, line in enumerate(lines[540:560], start=541):
    print(f"{i}: {line}", end='')

541:         epoch_loss = 0.0
542:         n_batches  = 0
543: 
544:         pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{args.epochs}", leave=False)
545:         for batch in pbar:
546:             optimizer.zero_grad()
547: 
548:             outputs = model(
549:                 input_ids=batch["input_ids"],
550:                 attention_mask=batch["attention_mask"],
551:                 pixel_values=batch["pixel_values"],
552:                 labels=batch["labels"],
553:             )
554:             loss = outputs.loss
555:             accelerator.backward(loss)
556: 
557:             accelerator.clip_grad_norm_(model.parameters(), 1.0)
558:             optimizer.step()
559:             scheduler.step()
560: 


In [ ]:
with open('/content/surgical-annotator/training/finetune.py', 'r') as f:
    lines = f.readlines()

# Fix model forward call to include token_type_ids
lines[547] = '            outputs = model(\n'
lines[548] = '                input_ids=batch["input_ids"],\n'
lines[549] = '                attention_mask=batch["attention_mask"],\n'
lines[550] = '                pixel_values=batch["pixel_values"],\n'
lines[551] = '                token_type_ids=batch.get("token_type_ids"),\n'
lines[552] = '                labels=batch["labels"],\n'
lines[553] = '            )\n'

with open('/content/surgical-annotator/training/finetune.py', 'w') as f:
    f.writelines(lines)

print("✓ Patched model forward call")

✓ Patched model forward call


In [ ]:
with open('/content/surgical-annotator/training/finetune.py', 'r') as f:
    lines = f.readlines()

for i, line in enumerate(lines[298:325], start=299):
    print(f"{i}: {line}", end='')

299: 
300:         # Full label tensor: -100 for image+prompt tokens, then answer tokens
301:         ignore = torch.full(
302:             (len(batch), input_len), -100, dtype=torch.long
303:         )
304:         labels = torch.cat([ignore, target_ids], dim=1)
305: 
306:         # Extend input_ids and attention_mask to include answer tokens
307:         # (teacher forcing: model sees the full sequence, predicts shifted)
308:         full_input_ids = torch.cat(
309:             [inputs["input_ids"], targets["input_ids"].masked_fill(
310:                 targets["input_ids"] == pad_id, 0)], dim=1
311:         )
312:         full_attn_mask = torch.cat(
313:             [inputs["attention_mask"], targets["attention_mask"]], dim=1
314:         )
315: 
316:         return {
317:             "input_ids":        full_input_ids,
318:             "attention_mask":   full_attn_mask,
319:             "pixel_values":     inputs["pixel_values"],
320:             "labels":           labels,
321:  

In [ ]:
with open('/content/surgical-annotator/training/finetune.py', 'r') as f:
    lines = f.readlines()

# Replace the return dict to include token_type_ids
lines[315] = '        # Build token_type_ids: 0 for image tokens, 1 for text tokens\n'
lines[316] = '        # PaliGemma requires this during training\n'
lines[317] = '        token_type_ids = None\n'
lines[318] = '        if "token_type_ids" in inputs:\n'
lines[319] = '            # Extend with 1s for the answer tokens\n'
lines[320] = '            answer_type_ids = torch.ones(\n'
lines[321] = '                len(batch), targets["input_ids"].shape[1], dtype=torch.long\n'
lines[322] = '            )\n'
lines[323] = '            token_type_ids = torch.cat(\n'
lines[324] = '                [inputs["token_type_ids"], answer_type_ids], dim=1\n'
lines[325] = '            )\n'
lines[326] = '\n'
lines[327] = '        return {\n'
lines[328] = '            "input_ids":        full_input_ids,\n'
lines[329] = '            "attention_mask":   full_attn_mask,\n'
lines[330] = '            "pixel_values":     inputs["pixel_values"],\n'
lines[331] = '            "token_type_ids":   token_type_ids,\n'
lines[332] = '            "labels":           labels,\n'
lines[333] = '            "label_names":      label_names,\n'
lines[334] = '        }\n'

with open('/content/surgical-annotator/training/finetune.py', 'w') as f:
    f.writelines(lines)

print("✓ Patched return dict with token_type_ids")

# Verify
with open('/content/surgical-annotator/training/finetune.py', 'r') as f:
    lines = f.readlines()
for i, line in enumerate(lines[313:337], start=314):
    print(f"{i}: {line}", end='')

✓ Patched return dict with token_type_ids
314:         )
315: 
316:         # Build token_type_ids: 0 for image tokens, 1 for text tokens
317:         # PaliGemma requires this during training
318:         token_type_ids = None
319:         if "token_type_ids" in inputs:
320:             # Extend with 1s for the answer tokens
321:             answer_type_ids = torch.ones(
322:                 len(batch), targets["input_ids"].shape[1], dtype=torch.long
323:             )
324:             token_type_ids = torch.cat(
325:                 [inputs["token_type_ids"], answer_type_ids], dim=1
326:             )
327: 
328:         return {
329:             "input_ids":        full_input_ids,
330:             "attention_mask":   full_attn_mask,
331:             "pixel_values":     inputs["pixel_values"],
332:             "token_type_ids":   token_type_ids,
333:             "labels":           labels,
334:             "label_names":      label_names,
335:         }
336:     """
337:     model.eva

In [ ]:
with open('/content/surgical-annotator/training/finetune.py', 'r') as f:
    content = f.read()

# Find and show what's around line 336 (the stray """)
lines = content.split('\n')
for i, line in enumerate(lines[330:345], start=331):
    print(f"{i}: {repr(line)}")

331: '            "pixel_values":     inputs["pixel_values"],'
332: '            "token_type_ids":   token_type_ids,'
333: '            "labels":           labels,'
334: '            "label_names":      label_names,'
335: '        }'
336: '    """'
337: '    model.eval()'
338: '    correct = 0'
339: '    total   = 0'
340: '    per_class_correct: dict[str, int] = {}'
341: '    per_class_total:   dict[str, int] = {}'
342: ''
343: '    # Pre-tokenise all organ names to get their first token id'
344: '    organ_first_tokens = []'
345: '    for organ in organ_list:'


In [ ]:
with open('/content/surgical-annotator/training/finetune.py', 'r') as f:
    content = f.read()

# Remove the stray """ that's breaking everything
# It appears right after the return dict closing brace
old = '''        }
    """
    model.eval()'''

new = '''        }
    return collate


def evaluate_model(model, processor, data_loader, device):
    model.eval()'''

if old in content:
    content = content.replace(old, new)
    print("✓ Fixed stray triple quote")
else:
    print("Pattern not found — showing context around line 335:")
    lines = content.split('\n')
    for i, line in enumerate(lines[330:345], start=331):
        print(f"{i}: {repr(line)}")

with open('/content/surgical-annotator/training/finetune.py', 'w') as f:
    f.write(content)

✓ Fixed stray triple quote


In [ ]:
with open('/content/surgical-annotator/training/finetune.py', 'r') as f:
    content = f.read()

# The return collate is missing and evaluate_model def is missing
# Check what's there now
old = '''        }
    return collate


def evaluate_model(model, processor, data_loader, device):
    model.eval()'''

if old in content:
    print("✓ Already correct — try running training now")
else:
    print("Still needs fixing — applying patch")
    # Find the closing brace and fix
    old2 = '''        }
    model.eval()'''
    new2 = '''        }
    return collate


def evaluate_model(model, processor, data_loader, device):
    model.eval()'''

    if old2 in content:
        content = content.replace(old2, new2)
        with open('/content/surgical-annotator/training/finetune.py', 'w') as f:
            f.write(content)
        print("✓ Fixed — return collate added")
    else:
        print("Pattern still not matching — showing lines 333-342:")
        lines = content.split('\n')
        for i, line in enumerate(lines[332:343], start=333):
            print(f"{i}: {repr(line)}")

✓ Already correct — try running training now


In [ ]:
with open('/content/surgical-annotator/training/finetune.py', 'r') as f:
    lines = f.readlines()

for i, line in enumerate(lines[544:562], start=545):
    print(f"{i}: {line}", end='')

545:         n_batches  = 0
546: 
547:         pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{args.epochs}", leave=False)
548:         for batch in pbar:
549:             optimizer.zero_grad()
550: 
551:             outputs = model(
552:                 input_ids=batch["input_ids"],
553:                 attention_mask=batch["attention_mask"],
554:                 pixel_values=batch["pixel_values"],
555:                 token_type_ids=batch.get("token_type_ids"),
556:                 labels=batch["labels"],
557:             )
558:             accelerator.backward(loss)
559: 
560:             accelerator.clip_grad_norm_(model.parameters(), 1.0)
561:             optimizer.step()
562:             scheduler.step()


In [ ]:
with open('/content/surgical-annotator/training/finetune.py', 'r') as f:
    lines = f.readlines()

# Line 558 (0-indexed: 557) should have loss = outputs.loss before backward
# Insert it before the backward call
lines.insert(557, '            loss = outputs.loss\n')

with open('/content/surgical-annotator/training/finetune.py', 'w') as f:
    f.writelines(lines)

print("✓ Patched loss assignment")

# Verify
with open('/content/surgical-annotator/training/finetune.py', 'r') as f:
    lines = f.readlines()
for i, line in enumerate(lines[549:563], start=550):
    print(f"{i}: {line}", end='')

✓ Patched loss assignment
550: 
551:             outputs = model(
552:                 input_ids=batch["input_ids"],
553:                 attention_mask=batch["attention_mask"],
554:                 pixel_values=batch["pixel_values"],
555:                 token_type_ids=batch.get("token_type_ids"),
556:                 labels=batch["labels"],
557:             )
558:             loss = outputs.loss
559:             accelerator.backward(loss)
560: 
561:             accelerator.clip_grad_norm_(model.parameters(), 1.0)
562:             optimizer.step()
563:             scheduler.step()


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import os

# Set memory config
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Patch gradient checkpointing into finetune.py
with open('/content/surgical-annotator/training/finetune.py', 'r') as f:
    content = f.read()

old = '    model = get_peft_model(model, lora_config)'
new = '''    model = get_peft_model(model, lora_config)
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()'''

if old in content:
    content = content.replace(old, new)
    with open('/content/surgical-annotator/training/finetune.py', 'w') as f:
        f.write(content)
    print("✓ Gradient checkpointing enabled")
else:
    print("⚠ Pattern not found — may already be patched")

# Update training config
import torch
EPOCHS       = 2
BATCH_SIZE   = 1
LR           = 2e-5
LORA_R       = 4
LORA_ALPHA   = 8
LORA_DROPOUT = 0.05
PATIENCE     = 3
TRAIN_RATIO  = 0.8
VAL_RATIO    = 0.1
BF16         = torch.cuda.is_available()

print(f"✓ Config updated: batch_size={BATCH_SIZE}, lora_r={LORA_R}")

# Rebuild command
REPO_DIR   = '/content/surgical-annotator'
MODEL_ID   = 'google/paligemma2-3b-pt-224'
OUTPUT_DIR = '/content/checkpoints'

DATASET_SPECS = [
    '/content/datasets/dsad_pilot/annotations.json:'
    '/content/datasets/dsad_pilot/images:3.0',
]

cmd = (
    f"python {REPO_DIR}/training/finetune.py "
    f"  --datasets {' '.join(DATASET_SPECS)} "
    f"  --output_dir {OUTPUT_DIR} "
    f"  --model_id {MODEL_ID} "
    f"  --epochs {EPOCHS} "
    f"  --batch_size {BATCH_SIZE} "
    f"  --learning_rate {LR} "
    f"  --lora_r {LORA_R} "
    f"  --lora_alpha {LORA_ALPHA} "
    f"  --lora_dropout {LORA_DROPOUT} "
    f"  --patience {PATIENCE} "
    f"  --train_ratio {TRAIN_RATIO} "
    f"  --val_ratio {VAL_RATIO} "
    + ('  --bf16' if BF16 else '')
)

print(f"✓ Command ready")
print(cmd.replace('  ', '\n  '))

✓ Gradient checkpointing enabled
✓ Config updated: batch_size=1, lora_r=4
✓ Command ready
python /content/surgical-annotator/training/finetune.py
   --datasets /content/datasets/dsad_pilot/annotations.json:/content/datasets/dsad_pilot/images:3.0
   --output_dir /content/checkpoints
   --model_id google/paligemma2-3b-pt-224
   --epochs 2
   --batch_size 1
   --learning_rate 2e-05
   --lora_r 4
   --lora_alpha 8
   --lora_dropout 0.05
   --patience 3
   --train_ratio 0.8
   --val_ratio 0.1
   --bf16


In [ ]:
with open('/content/surgical-annotator/training/finetune.py', 'r') as f:
    content = f.read()

# Check what's around line 575
lines = content.split('\n')
for i, line in enumerate(lines[568:582], start=569):
    print(f"{i}: {line}")

569:             pbar.set_postfix(loss=f"{loss.item():.4f}")
570: 
571:         avg_loss = epoch_loss / max(n_batches, 1)
572: 
573:         # Validation
574:         from PIL import Image
575:         val_metrics = evaluate(model, processor, val_loader, accelerator, organ_list)
576:         val_acc = val_metrics["accuracy"]
577: 
578:         logger.info(
579:             "Epoch %d/%d — loss: %.4f | val_acc: %.3f",
580:             epoch, args.epochs, avg_loss, val_acc,
581:         )
582: 


In [ ]:
with open('/content/surgical-annotator/training/finetune.py', 'r') as f:
    content = f.read()

# Fix the call to match the actual function name
old = 'val_metrics = evaluate(model, processor, val_loader, accelerator, organ_list)'
new = 'val_metrics = evaluate_model(model, processor, val_loader, accelerator, organ_list)'

if old in content:
    content = content.replace(old, new)
    with open('/content/surgical-annotator/training/finetune.py', 'w') as f:
        f.write(content)
    print("✓ Fixed evaluate → evaluate_model")
else:
    print("⚠ Pattern not found")

✓ Fixed evaluate → evaluate_model


In [10]:
with open('/content/surgical-annotator/training/finetune.py', 'r') as f:
    content = f.read()

# Check if os is imported at top
if content.startswith('import os'):
    print("✓ os already imported")
else:
    content = 'import os\n' + content
    with open('/content/surgical-annotator/training/finetune.py', 'w') as f:
        f.write(content)
    print("✓ Added import os")

✓ Added import os


In [12]:
with open('/content/surgical-annotator/training/finetune.py', 'r') as f:
    lines = f.readlines()

# Show lines around 463-470 to see exact current state
for i, line in enumerate(lines[460:472], start=461):
    print(f"{i}: {repr(line)}")

461: '        torch_dtype=torch.bfloat16 if args.bf16 else torch.float32,\n'
462: '    )\n'
463: '\n'
464: '    # Freeze vision tower — only fine-tune the language model layers with LoRA\n'
465: '    for param in model.vision_tower.parameters():\n'
466: '        param.requires_grad = False\n'
467: '    for param in model.multi_modal_projector.parameters():\n'
468: '        param.requires_grad = False\n'
469: '\n'
470: '    # LoRA on language model attention layers\n'
471: '    lora_config = LoraConfig(\n'
472: '        r=args.lora_r,\n'


In [17]:
import subprocess
subprocess.run(['pip', 'install', '--upgrade', 'torchao', '-q'], check=True)
print("✓ torchao upgraded")

✓ torchao upgraded


In [19]:
with open('/content/surgical-annotator/training/finetune.py', 'r') as f:
    content = f.read()

content = content.replace(
    'val_metrics = evaluate(model, processor, val_loader, accelerator, organ_list)',
    'val_metrics = evaluate_model(model, processor, val_loader, accelerator, organ_list)'
)

with open('/content/surgical-annotator/training/finetune.py', 'w') as f:
    f.write(content)

print("✓ Fixed")

✓ Fixed


In [23]:
# ⚠ This cell runs the fine-tuning — may take 30 min – 3 hours depending
# on dataset size, GPU, and number of epochs.

import subprocess, sys

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    pass

env = os.environ.copy()
if hf_token:
    env['HF_TOKEN'] = hf_token
env['PYTHONPATH'] = f"{REPO_DIR}/training:{env.get('PYTHONPATH', '')}"

proc = subprocess.Popen(
    cmd,
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=env,
)

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
print(f'\n--- finetune.py exited with code {proc.returncode} ---')
if proc.returncode != 0:
    print('Training FAILED. Check the output above for errors.')
else:
    print('✓ Training complete!')

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
2026-06-05 16:31:47,134  INFO      Device: cuda | Mixed precision: bf16
2026-06-05 16:31:47,134  INFO      Building WeightedSurgicalDataset from 1 spec(s)...
2026-06-05 16:31:47,135  INFO      Loading dataset 'dsad_pilot' from /content/datasets/dsad_pilot/annotations.json (weight=3.00)...
2026-06-05 16:31:47,149  INFO      Dataset: 257 samples across 6 classes from 1 export file(s)
2026-06-05 16:31:47,149  INFO        stomach                 112 samples
2026-06-05 16:31:47,149  INFO        pancreas                 46 samples ⚠ LOW
2026-06-05 16:31:47,149  INFO        colon           

In [22]:
with open('/content/surgical-annotator/training/finetune.py', 'r') as f:
    content = f.read()

content = content.replace('def evaluate(', 'def evaluate_model(')

with open('/content/surgical-annotator/training/finetune.py', 'w') as f:
    f.write(content)

print("✓ Fixed — renamed def evaluate → def evaluate_model")

✓ Fixed — renamed def evaluate → def evaluate_model


In [21]:
with open('/content/surgical-annotator/training/finetune.py', 'r') as f:
    content = f.read()

# Check if function exists
if 'def evaluate_model' in content:
    print("✓ evaluate_model function exists")
elif 'def evaluate' in content:
    print("✓ evaluate function exists (wrong name)")
else:
    print("✗ no evaluate function found at all")

# Check the call
if 'val_metrics = evaluate_model' in content:
    print("✓ call uses evaluate_model")
elif 'val_metrics = evaluate(' in content:
    print("✗ call still uses evaluate")

✓ evaluate function exists (wrong name)
✓ call uses evaluate_model


---
## 5 · Evaluate

In [24]:
EVAL_DATASET_SPEC = DATASET_SPECS[0]  # evaluate on first dataset by default
parts    = EVAL_DATASET_SPEC.split(':')
EVAL_JSON    = parts[0]
EVAL_IMAGES  = parts[1]
CHECKPOINT   = f'{OUTPUT_DIR}/best_model'
EVAL_OUT_DIR = '/content/evaluation_results'

eval_cmd = (
    f"python {REPO_DIR}/training/evaluate.py "
    f"  --checkpoint {CHECKPOINT} "
    f"  --coco_exports {EVAL_JSON} "
    f"  --images_dir {EVAL_IMAGES} "
    f"  --output_dir {EVAL_OUT_DIR}"
)

print('Running evaluation...')
!{eval_cmd}

# Show confusion matrix if it was generated
cm_path = f'{EVAL_OUT_DIR}/confusion_matrix.png'
if os.path.exists(cm_path):
    from IPython.display import Image as IPyImage, display
    display(IPyImage(filename=cm_path))

# Show metrics
metrics_path = f'{EVAL_OUT_DIR}/metrics.json'
if os.path.exists(metrics_path):
    import json
    with open(metrics_path) as f:
        metrics = json.load(f)
    print('\nMetrics:')
    print(json.dumps(metrics, indent=2))

Running evaluation...
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
2026-06-05 16:50:59,549  INFO      Using device: cuda
2026-06-05 16:50:59,566  INFO      Dataset: 257 samples across 6 classes from 1 export file(s)
2026-06-05 16:50:59,566  INFO        stomach                 112 samples
2026-06-05 16:50:59,566  INFO        pancreas                 46 samples ⚠ LOW
2026-06-05 16:50:59,566  INFO        colon                    44 samples ⚠ LOW
2026-06-05 16:50:59,566  INFO        abdominal_wall           37 samples ⚠ LOW
2026-06-05 16:50:59,566  INFO        small_intestine          16 samples ⚠ LOW
2026-06-05 16:50:59,566  INFO 

---
## 6 · Export merged model for production

In [25]:
PRODUCTION_DIR = '/content/production_model'

export_cmd = (
    f"python {REPO_DIR}/training/export_model.py "
    f"  --checkpoint {OUTPUT_DIR}/best_model "
    f"  --output {PRODUCTION_DIR}"
)

print('Exporting merged model...')
!{export_cmd}
print(f'✓ Production model saved to {PRODUCTION_DIR}')

# Show file sizes
!du -sh {PRODUCTION_DIR}/*

Exporting merged model...
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
2026-06-05 16:52:25,485  INFO      Device: cuda
2026-06-05 16:52:25,486  INFO      Loading base model: google/paligemma2-3b-pt-224
2026-06-05 16:52:25,830  INFO      HTTP Request: HEAD https://huggingface.co/google/paligemma2-3b-pt-224/resolve/main/config.json "HTTP/1.1 200 OK"
2026-06-05 16:52:26,075  INFO      HTTP Request: HEAD https://huggingface.co/google/paligemma2-3b-pt-224/resolve/main/config.json "HTTP/1.1 200 OK"
2026-06-05 16:52:26,434  INFO      HTTP Request: HEAD https://huggingface.co/google/paligemma2-3b-pt-224/resolve/main/model.safetensors "

---
## 7 · Download or push to HuggingFace Hub

In [ ]:
# Option A: Download checkpoints as a ZIP to your local machine
import shutil
from google.colab import files

zip_path = '/content/checkpoints.zip'
shutil.make_archive('/content/checkpoints', 'zip', OUTPUT_DIR)
print(f'Created {zip_path}')
files.download(zip_path)
print('✓ Download started')

In [ ]:
# Option B: Push production model directly to HuggingFace Hub
# Fill in HF_REPO_ID before running.

HF_REPO_ID = 'your-hf-username/surgical-organ-classifier'  # ← edit

from huggingface_hub import HfApi
api = HfApi()

try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None

api.upload_folder(
    folder_path=PRODUCTION_DIR,
    repo_id=HF_REPO_ID,
    repo_type='model',
    token=hf_token,
    commit_message='Fine-tuned surgical organ classifier',
)
print(f'✓ Model pushed to https://huggingface.co/{HF_REPO_ID}')

---
## 8 · Quick inference test

In [ ]:
import json
from PIL import Image
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration
from peft import PeftModel
import torch

CHECKPOINT = f'{OUTPUT_DIR}/best_model'
PROMPT = '<image> Identify the highlighted surgical organ. Answer with the organ name only:'

print('Loading fine-tuned model...')
processor = AutoProcessor.from_pretrained(CHECKPOINT)
model = PaliGemmaForConditionalGeneration.from_pretrained(
    CHECKPOINT,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
)
model.eval()

# Load organ list written during training
organ_list_path = f'{OUTPUT_DIR}/organ_list.json'
if os.path.exists(organ_list_path):
    with open(organ_list_path) as f:
        organ_list = json.load(f)
    print(f'Organ classes: {organ_list}')

# Test with a sample image from DSAD
dsad_images = sorted([
    f for f in os.listdir(f'{DATASETS_ROOT}/dsad/images')
    if f.endswith('.jpg')
])[:5]

for fname in dsad_images:
    img_path = f'{DATASETS_ROOT}/dsad/images/{fname}'
    image    = Image.open(img_path).convert('RGB')

    inputs = processor(
        images=image,
        text=PROMPT,
        return_tensors='pt',
    ).to(model.device)

    with torch.no_grad():
        generated = model.generate(**inputs, max_new_tokens=8, do_sample=False)

    input_len = inputs['input_ids'].shape[1]
    pred = processor.decode(generated[0][input_len:], skip_special_tokens=True).strip()
    print(f'  {fname}  →  "{pred}"')

print('\n✓ Inference test complete')

---
## Notes

**Memory tips for T4 (15 GB VRAM)**
- If you hit OOM, set `BATCH_SIZE = 4`.
- LoRA `r=8` uses ~40% less memory than the default `r=16`.

**Enable BF16**  
Automatically enabled if a GPU is detected.  Halves VRAM usage and speeds up training ~2×.

**Resuming from a checkpoint**  
Set `CHECKPOINT` to point to `checkpoints/latest` and add `--resume_from_checkpoint` support in `finetune.py` (coming soon).

**Deploying the model**  
After export, copy `production_model/` to your server and set:
```bash
export VLM_BACKEND=finetuned
export FINETUNED_MODEL_PATH=./training/production_model
```